In [ ]:
# importing necessary python modules for pyspark execution
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,\
                            trim, upper, avg, round,\
                            sum as spark_sum, to_date, when, count,\
                            current_timestamp, regexp_replace, regexp,\
                            broadcast, round as spark_round, max as spark_max, mode,\
                            lit, desc, collect_list, min as spark_min, median as spark_median
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType, FloatType

In [ ]:
##################################################################################################################
###################################### SparkSession Creation function ############################################
########################## This helps to create sparksession #####################################################
##################################################################################################################

In [ ]:
# Create helper static fucntion for creating spark session
@staticmethod
def create_spark_session():
    spark = SparkSession.builder\
                         .appName('Mini_Assignment1_Session')\
                         .getOrCreate()
    return spark

In [ ]:
##################################################################################################################
###################################### Metadata helper class #####################################################
########################## This helps to get the metadta of the dataframe ########################################
##################################################################################################################

In [ ]:
# Lets create a helper class which provide metadata information of the dataframe
class extractMetadata:
    """
    This class will get dataframe as input and extract its metadata information
    """
    def __init__(self, df):
        """
        Initialize with dataframe --> used in getting the metadata
        Initialize with dataframe's row count and columns --> which provides the dataframe shape
        """
        self.df = df
        self.row_count = df.count()
        self.columns = df.columns
    # Below function will get the dataframe shape and return that as tuple (row_count, column_count)
    def _get_dataframe_shape(self):
        return (self.row_count, len(self.columns))
    # Below function will get the column names and store that in a python list
    def _get_column_names(self):
        return self.df.columns
    # Below function will get the column data types and store that in a python list
    def _get_column_types(self):
        return [datatype[1] for datatype in self.df.dtypes]
    # Below function will get the missing values count for each column and store that in a python list
    def _get_missing_column_counts(self):
        return [self.df.filter(col(column).isNull()).count() for column in self.columns]
    # Below function will get the missing values proportion for each column and store that in a python list
    def _get_missing_column_proportion(self):
        return [self.df.filter(col(column).isNull()).count()/self.row_count for column in self.columns]
    # Below function will get the distinct count of each column and store that in a python list
    def _get_unique_column_counts(self):
        return [self.df.select(col(column)).groupBy(col(column)).count().select(column).count() for column in self.columns]
    # Below function will get index column possibility and store that in a python list
    def _index_column_possiblity(self):
        return ["All Unique" if self.df.select(col(column)).groupBy(col(column)).count().select(column).count() == self.row_count else "Non Unique Column" for column in self.columns]
    # Below function will get the categorical columns top 3 mode value and store that in a python list
    def _get_categorical_mode_values(self):
        categorical_list = []
        for field in self.df.schema.fields:
            if field.dataType not in (DoubleType(),IntegerType()):
                top_3_rows = (
                self.df.groupBy(col(field.name))
                .count()
                .orderBy(desc('count'))
                .head(3)
                )
                top_3_values = [f'{row[0]}-{row[1]}' for row in top_3_rows]
                categorical_list.append(top_3_values)
            else:
                categorical_list.append(['Not a categorical column'])
        return categorical_list
    # Below function will get the numerical columns min, max, median value and store that in a python list
    def _get_numerical_distribution_values(self):
        numerical_list = []
        for field in self.df.schema.fields:
            if field.dataType in (DoubleType(),IntegerType()):
                dist_dict = ([
                            f'min - {self.df.select(spark_min(field.name)).collect()[0][0]}, median - {self.df.select(spark_median(field.name)).collect()[0][0]}, max - {self.df.select(spark_max(field.name)).collect()[0][0]}'
                            ])
                numerical_list.append(dist_dict)
            else:
                numerical_list.append(([f'{field.name}: Not a numerical column']))
        single_column_data = [(item,) for item in numerical_list]
        return single_column_data
    def extract_metadata(self, spark):
        metadata = list(zip(self._get_column_names(),
                            self._get_column_types(),
                            self._get_missing_column_counts(),
                            self._get_missing_column_proportion(),
                            self._get_unique_column_counts(),
                            self._index_column_possiblity(),
                            self._get_categorical_mode_values(),
                            self._get_numerical_distribution_values()
                           )
                       )
        columns = ["column_name",
                   "data_type",
                   "missing_column_count",
                   "missing_column_proportion(%)",
                   "unique_column_count",
                   "index_column_possibility",
                   "top_3_values",
                   "numerical_column_distribution"
                  ]
        return spark.createDataFrame(metadata,
                                    columns)
        
        

In [ ]:
### Read your dataframe from csv/tsv/or any other source ###
## df = your code to read the files

In [ ]:
# Read the metadata using metadata helper class and keep this ready in the dataframe for future analysis
df_metadata = extractMetadata(df).extract_metadata(spark)